In [2]:
# HARE KRISHNA
# Final Training Script - Resilient & Robust - V2 FIXED + ENHANCED

!pip -qq install torch torchvision transformers opencv-python-headless numpy matplotlib tqdm seaborn

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import TimesformerForVideoClassification
import cv2
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
import torchvision.transforms as transforms
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Updated AMP import with fallback
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = 'cuda'
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = None

#from google.colab import drive
#drive.mount('/content/drive')

# Set environment variable to avoid memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- CONFIGURATION ---
DATASET_PATH = "/content/drive/MyDrive/SceneSolverG388/TimeSformermerAnomalyMergedClassification/MergedAnomalyDetection"
MODELS_PATH = "/content/drive/MyDrive/SceneSolverG388/TimeSformermerAnomalyMergedClassification/Models_7_Class_V2_Resilient"
NUM_EPOCHS = 30
LEARNING_RATE = 3e-5
BATCH_SIZE = 2
NUM_FRAMES = 16

# Define classes (7 classes)
NUM_CLASSES = 7
class_names = {
    0: "Aggression", 1: "TheftOrLarceny", 2: "Firebombing",
    3: "LawEnforcement", 4: "Shooting", 5: "Vandalism", 6: "RoadAccidents"
}

# --- PATHS ---
CHECKPOINT_PATH = os.path.join(MODELS_PATH, "Checkpoints")
RESULTS_PATH = os.path.join(MODELS_PATH, "Results")
BEST_MODEL_PATH = os.path.join(MODELS_PATH, "best_model.pth")
LATEST_CHECKPOINT_FILE = os.path.join(CHECKPOINT_PATH, "latest_checkpoint.pth")
os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)


# === BEST MODEL RECOVERY ===
def recover_best_model_from_checkpoint(device):
    """
    If best_model.pth is missing but a latest checkpoint exists,
    extract and save model weights from the checkpoint automatically.
    """
    if not os.path.exists(BEST_MODEL_PATH) and os.path.exists(LATEST_CHECKPOINT_FILE):
        print("⚠️  Best model not found. Recovering from latest checkpoint...")
        ckpt = torch.load(LATEST_CHECKPOINT_FILE, map_location=device)
        torch.save(ckpt['model_state_dict'], BEST_MODEL_PATH)
        best_acc = ckpt.get('history', {}).get('best_val_acc', 0.0)
        print(f"✅  Best model recovered (recorded acc: {best_acc:.2f}%)")
        return best_acc
    return None


# --- TRANSFORMS ---
def get_transforms():
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.1, scale=(0.02, 0.15))
    ])
    val_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform


# --- DATASET ---
class AnomalyDetectionDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None, num_frames=16, frame_size=(224, 224)):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.num_frames = num_frames
        self.frame_size = frame_size

    def __len__(self):
        return len(self.video_paths)

    def get_labels(self):
        """Return label list instantly — no disk I/O. Used for class weighting."""
        return self.labels

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        try:
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                return None, None

            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames < self.num_frames:
                cap.release()
                return None, None

            if total_frames > self.num_frames * 2:
                start_frame = total_frames // 10
                end_frame = total_frames - (total_frames // 10)
                frame_indices = np.linspace(start_frame, end_frame - 1, self.num_frames, dtype=int)
            else:
                frame_indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)

            frames = []
            for frame_idx in frame_indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
                ret, frame = cap.read()
                if ret:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = cv2.resize(frame, self.frame_size)
                    frames.append(frame)

            cap.release()

            if len(frames) < self.num_frames:
                return None, None

            if self.transform:
                frames = [self.transform(frame) for frame in frames]

            return torch.stack(frames), torch.tensor(label, dtype=torch.long)

        except Exception as e:
            print(f"Error processing {video_path}: {e}")
            return None, None


# --- COLLATE: skip None samples silently ---
def custom_collate_fn(batch):
    batch = [item for item in batch if item[0] is not None]
    if not batch:
        return None, None
    frames, labels = zip(*batch)
    return torch.stack(frames), torch.stack(labels)


# --- DATASET LOADING ---
def load_dataset_paths(dataset_path):
    categories = list(class_names.values())
    video_paths = []
    labels = []

    for idx, category in enumerate(categories):
        category_path = os.path.join(dataset_path, category)
        if not os.path.exists(category_path):
            print(f"Warning: Directory not found for '{category}'")
            continue
        for video_file in os.listdir(category_path):
            if video_file.lower().endswith('.mp4'):
                video_paths.append(os.path.join(category_path, video_file))
                labels.append(idx)

    if len(video_paths) > 0:
        video_paths, labels = balance_classes(video_paths, labels)

    return video_paths, labels


def balance_classes(video_paths, labels, target_samples=300):
    """
    Balance classes by duplicating underrepresented samples.
    Operates only on path strings — zero disk I/O, instant.
    """
    class_videos = defaultdict(list)
    for path, label in zip(video_paths, labels):
        class_videos[label].append(path)

    balanced_paths = []
    balanced_labels = []

    print("\n--- Class Balancing ---")
    for class_idx in range(len(class_names)):
        if class_idx not in class_videos:
            print(f"{class_names[class_idx]}: MISSING — skipping")
            continue

        current_videos = class_videos[class_idx]
        current_count = len(current_videos)
        target_count = min(target_samples, max(current_count, 150))

        if current_count >= target_count:
            selected_videos = current_videos
        else:
            needed = target_count - current_count
            selected_videos = current_videos.copy()
            for _ in range(needed):
                selected_videos.append(random.choice(current_videos))

        balanced_paths.extend(selected_videos)
        balanced_labels.extend([class_idx] * len(selected_videos))
        print(f"  {class_names[class_idx]}: {current_count} → {len(selected_videos)} videos")

    print(f"Total videos after balancing: {len(balanced_paths)}")
    return balanced_paths, balanced_labels


# --- MODEL ---
class TimeSformerAnomalyClassifier(nn.Module):
    def __init__(self, num_classes):
        super(TimeSformerAnomalyClassifier, self).__init__()
        self.timesformer = TimesformerForVideoClassification.from_pretrained(
            "facebook/timesformer-base-finetuned-k400",
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )

        hidden_size = self.timesformer.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, num_classes)
        )
        self.timesformer.classifier = self.classifier

    def forward(self, x):
        return self.timesformer(pixel_values=x).logits


# --- TRAINING LOOP ---
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, start_epoch, history):
    best_val_acc = history.get('best_val_acc', 0.0)
    train_losses = history.get('train_losses', [])
    val_losses = history.get('val_losses', [])
    train_accuracies = history.get('train_accuracies', [])
    val_accuracies = history.get('val_accuracies', [])

    patience = 7
    patience_counter = history.get('patience_counter', 0)

    # AMP scaler
    if AMP_DEVICE:
        scaler = GradScaler(AMP_DEVICE)
    else:
        scaler = GradScaler()

    for epoch in range(start_epoch, num_epochs):
        print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")

        # ---- TRAIN ----
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_pbar = tqdm(train_loader, desc="Training")
        for frames, labels in train_pbar:
            # Skip empty batches from collate_fn
            if frames is None:
                continue

            frames, labels = frames.to(device), labels.to(device)
            optimizer.zero_grad()

            ctx = autocast(AMP_DEVICE) if AMP_DEVICE else autocast()
            with ctx:
                outputs = model(frames)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * frames.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            current_acc = (100 * train_correct / train_total) if train_total > 0 else 0
            train_pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})

        epoch_train_loss = train_loss / train_total if train_total > 0 else 0
        epoch_train_acc = (100 * train_correct / train_total) if train_total > 0 else 0
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

        # ---- VALIDATE ----
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc="Validation")
            for frames, labels in val_pbar:
                if frames is None:
                    continue

                frames, labels = frames.to(device), labels.to(device)
                ctx = autocast(AMP_DEVICE) if AMP_DEVICE else autocast()
                with ctx:
                    outputs = model(frames)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * frames.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                current_acc = (100 * val_correct / val_total) if val_total > 0 else 0
                val_pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})

        epoch_val_loss = val_loss / val_total if val_total > 0 else 0
        epoch_val_acc = (100 * val_correct / val_total) if val_total > 0 else 0
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)
        print(f"Validation Loss: {epoch_val_loss:.4f} | Validation Acc: {epoch_val_acc:.2f}%")

        # ---- LR SCHEDULER ----
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(epoch_val_loss)
        else:
            scheduler.step()
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")

        # ---- EARLY STOPPING COUNTER ----
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
        else:
            patience_counter += 1

        # ---- SAVE CHECKPOINT (every epoch) ----
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': {
                'train_losses': train_losses,
                'train_accuracies': train_accuracies,
                'val_losses': val_losses,
                'val_accuracies': val_accuracies,
                'best_val_acc': best_val_acc,
                'patience_counter': patience_counter,   # persist across resume
            }
        }
        torch.save(checkpoint, LATEST_CHECKPOINT_FILE)
        print(f"Checkpoint saved (epoch {epoch + 1})")

        # ---- SAVE BEST MODEL ----
        # FIX: was checking `>= best_val_acc` after updating best_val_acc above,
        # so it saved on every epoch. Now correctly saves only on genuine improvement.
        if patience_counter == 0:
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"✅  New best model saved — acc: {best_val_acc:.2f}%")

            report = classification_report(
                all_labels, all_preds,
                target_names=list(class_names.values()),
                output_dict=True
            )
            with open(os.path.join(RESULTS_PATH, f"classification_report_epoch_{epoch + 1}.json"), 'w') as f:
                json.dump(report, f, indent=4)

            cm = confusion_matrix(all_labels, all_preds)
            plt.figure(figsize=(12, 10))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=list(class_names.values()),
                        yticklabels=list(class_names.values()),
                        cbar_kws={'label': 'Count'})
            plt.title(f'Confusion Matrix — Epoch {epoch + 1} (Acc: {epoch_val_acc:.2f}%)')
            plt.xlabel('Predicted')
            plt.ylabel('True')
            plt.tight_layout()
            plt.savefig(os.path.join(RESULTS_PATH, f"confusion_matrix_epoch_{epoch + 1}.png"),
                        dpi=300, bbox_inches='tight')
            plt.close()

        # ---- EARLY STOPPING ----
        if patience_counter >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break

    print("\nTraining finished.")
    _plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies)


def _plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies):
    """Save training curves to disk."""
    epochs_range = range(1, len(train_losses) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, train_losses, label='Train Loss')
    plt.plot(epochs_range, val_losses, label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, train_accuracies, label='Train Acc')
    plt.plot(epochs_range, val_accuracies, label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PATH, "training_history.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Training history plot saved to {RESULTS_PATH}/training_history.png")


# --- MAIN ---
def main():
    # Recover best model if lost
    recover_best_model_from_checkpoint(device)

    print("Loading dataset...")
    video_paths, labels = load_dataset_paths(DATASET_PATH)
    if not video_paths:
        print("FATAL: No videos found. Check DATASET_PATH and directory structure.")
        return

    print(f"\nFound {len(video_paths)} videos across {len(set(labels))} classes")
    unique_labels, counts = np.unique(labels, return_counts=True)
    for label, count in zip(unique_labels, counts):
        print(f"  {class_names[label]}: {count} videos")

    # Stratified splits
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        video_paths, labels, test_size=0.3, random_state=42, stratify=labels)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels)

    print(f"\nDataset splits — Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

    train_transform, val_transform = get_transforms()
    train_dataset = AnomalyDetectionDataset(train_paths, train_labels, transform=train_transform, num_frames=NUM_FRAMES)
    val_dataset   = AnomalyDetectionDataset(val_paths,   val_labels,   transform=val_transform,   num_frames=NUM_FRAMES)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, collate_fn=custom_collate_fn, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, collate_fn=custom_collate_fn, pin_memory=True)

    print("\nInitializing model...")
    model = TimeSformerAnomalyClassifier(num_classes=NUM_CLASSES).to(device)

    total_params    = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Class-weighted loss to handle any residual imbalance after balancing
    class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
    # Guard against zero counts for missing classes
    class_weights = torch.tensor(
        [sum(class_counts) / max(c, 1) for c in class_counts],
        dtype=torch.float
    ).to(device)
    print(f"Class weights: {class_weights.cpu().numpy()}")

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2, eps=1e-8)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    # Resume from checkpoint
    start_epoch = 0
    history = {}
    if os.path.exists(LATEST_CHECKPOINT_FILE):
        print(f"\nResuming from checkpoint: {LATEST_CHECKPOINT_FILE}")
        checkpoint = torch.load(LATEST_CHECKPOINT_FILE, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        history = checkpoint.get('history', {})
        print(f"Resuming from epoch {start_epoch}")
        if 'best_val_acc' in history:
            print(f"Previous best validation accuracy: {history['best_val_acc']:.2f}%")
    else:
        print("No checkpoint found. Starting from scratch.")

    print("\nStarting training...")
    train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=NUM_EPOCHS, start_epoch=start_epoch, history=history)


if __name__ == "__main__":
    main()

Using device: cuda
⚠️  Best model not found. Recovering from latest checkpoint...
✅  Best model recovered (recorded acc: 84.06%)
Loading dataset...

--- Class Balancing ---
  Aggression: 159 → 159 videos
  TheftOrLarceny: 451 → 451 videos
  Firebombing: 129 → 150 videos
  LawEnforcement: 55 → 150 videos
  Shooting: 73 → 150 videos
  Vandalism: 55 → 150 videos
  RoadAccidents: 173 → 173 videos
Total videos after balancing: 1383

Found 1383 videos across 7 classes
  Aggression: 159 videos
  TheftOrLarceny: 451 videos
  Firebombing: 150 videos
  LawEnforcement: 150 videos
  Shooting: 150 videos
  Vandalism: 150 videos
  RoadAccidents: 173 videos

Dataset splits — Train: 968, Val: 207, Test: 208

Initializing model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

...

Validation Loss: 1.2207 | Validation Acc: 82.14%
Current LR: 1.88e-06
Checkpoint saved (epoch 29)

--- Epoch 30/30 ---


Training: 100%|██████████| 484/484 [04:37<00:00,  1.75it/s, Loss=0.5597, Acc=100.00%]


Train Loss: 0.6714 | Train Acc: 100.00%


Validation: 100%|██████████| 104/104 [00:41<00:00,  2.48it/s, Loss=4.7423, Acc=84.29%]


Validation Loss: 1.1791 | Validation Acc: 84.29%
Current LR: 1.88e-06
Checkpoint saved (epoch 30)

Training finished.
Training history plot saved to /content/drive/MyDrive/SceneSolverG388/TimeSformermerAnomalyMergedClassification/Models_7_Class_V2_Resilient/Results/training_history.png
